# 01 — Data profiling

The first notebook of any project is not analysis; it is a look at what you actually received.
This one answers: how many rows, what is missing, what is the date range, and what will break
downstream.

Run from the repo root. It reads the CSVs in `data/raw/` — no database required.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

from src import config, transform
from src.utils import md_table

In [ ]:
raw = transform.read_raw()
for name, df in raw.items():
    print(f"{name:<20} {len(df):>9,} rows  {df.shape[1]:>3} cols")

## Row counts and grain

The number that matters is not the row count, it is *one row per what*. Note that
`orders` has one row per order while `customers` has one row per order-customer link —
`customer_id` is not a person.

In [ ]:
orders = raw["orders"]
print("orders            :", len(orders))
print("distinct order_id :", orders["order_id"].nunique())
print("distinct customer_id:", raw["customers"]["customer_id"].nunique())
print("distinct customer_unique_id:", raw["customers"]["customer_unique_id"].nunique())
print()
print(orders["order_status"].value_counts())

`customer_unique_id` is smaller than `customer_id` — that gap *is* the repeat-purchase behaviour.
Everything in the retention analysis depends on using the right one.

In [ ]:
purchase = pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce")
print("first order:", purchase.min())
print("last order :", purchase.max())
print()
monthly = purchase.dt.to_period("M").value_counts().sort_index()
monthly.head(6)

The first months are launch noise. That is why `config.ANALYSIS_START_MONTH` is `2017-01` —
a documented decision, not silent data loss.

In [ ]:
nulls = pd.DataFrame({
    "nulls": orders.isna().sum(),
    "pct": (orders.isna().mean() * 100).round(2),
}).sort_values("nulls", ascending=False)
nulls

## The delivery-date problem

`order_delivered_customer_date` is null for anything not delivered. If you compute
`AVG(delivery_days)` without filtering, the average silently excludes those orders and you will
report a better number than reality.

In [ ]:
delivered = orders[orders["order_status"] == "delivered"]
print("delivered orders:", len(delivered))
print("missing delivery date:",
      delivered["order_delivered_customer_date"].isna().sum())
print()
print("status distribution of orders with no delivery date:")
print(orders[orders["order_delivered_customer_date"].isna()]["order_status"].value_counts())

## Duplicates and joins that multiply

Two things to check before any join: is the join key unique on the right side, and does the
geolocation table have many rows per zip prefix?

In [ ]:
reviews = raw["order_reviews"]
print("review rows          :", len(reviews))
print("distinct review_id   :", reviews["review_id"].nunique())
print("distinct order_id    :", reviews["order_id"].nunique())
print()
geo = raw["geolocation"]
print("geolocation rows per zip prefix (top 5):")
print(geo.groupby("geolocation_zip_code_prefix").size().sort_values(ascending=False).head())

## Next

Run the automated gate for the full picture:

```bash
python -m src.quality.run_quality_checks
```

It writes `reports/data_quality_report.md` and exits non-zero if anything critical is broken.